# Project Overview: Reliability-Centered Early Toxicity Screening

## Scientific objective
Define the intended context of use, endpoint-specific decisions, user input/output contract, non-replacement disclaimer, and complete workflow.

## Inputs
- `README.md`
- `configs/endpoints.yaml`

## Expected outputs
- `reports/project_scope.json`
- `reports/workflow.mmd`
- `figures/project_workflow.png`

## Dependencies
Python, matplotlib, PyYAML

## Reproducibility seed
`20260723`. The seed is loaded from `configs/training_config.yaml`; split files and checkpoints are persisted.

## Data and model assumptions
SMILES is the primary structure input. A prediction is supported only when calibration, uncertainty, and applicability-domain checks are acceptable.

## Validation checks
The executable cells below fail explicitly on missing/inconsistent required artifacts and save machine-readable status records.

## Interpretation of results
Interpret endpoint-level outputs only after checking prevalence, missingness, split integrity, calibration, uncertainty, and applicability-domain coverage. No notebook result is evidence that experimental toxicity testing can be replaced.

## Saved artifacts
Artifacts listed above are written under `data/`, `models/`, `results/`, `figures/`, `tables/`, or `reports/` and are consumed by later notebooks.

## Limitations
Endpoint labels are assay proxies with heterogeneous evidence. The project is not a regulatory model and does not infer causal mechanisms.

## Next notebook
[01_environment_and_reproducibility.ipynb](./01_environment_and_reproducibility.ipynb)


### Context of use and endpoint decisions

The input is one SMILES string or a CSV containing molecule identifiers and SMILES. Each endpoint remains a distinct decision: hERG blockade, Ames mutagenicity, SR-p53, SR-ATAD5, SR-ARE, and SR-MMP. Output probabilities are research estimates that require experimental confirmation.

```mermaid
flowchart LR
  A[Known molecules and assay labels] --> B[Molecular standardization]
  B --> C[Quality control]
  C --> D[Chemical-space analysis]
  D --> E[Scaffold-aware splitting]
  E --> F[Baseline and advanced modeling]
  F --> G[Endpoint-specific evaluation]
  G --> H[Calibration and uncertainty]
  H --> I[Applicability domain and OOD]
  I --> J[Interpretability]
  J --> K[Candidate-screening interface]
```


In [1]:
import sys
print(sys.executable)

import toxicity_screening
print(toxicity_screening.__file__)

D:\Users\anaconda3\envs\toxicity-screening\python.exe
D:\Dropbox\Work\Learning\Python\toxicity_screening_project\src\toxicity_screening\__init__.py


In [2]:
from pathlib import Path
import os, json, warnings
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run this notebook from the repository root or notebooks directory")
os.chdir(ROOT)

from toxicity_screening.config import load_configs, execution_profile
from toxicity_screening.utils import set_global_seed, require_paths

CONFIGS = load_configs(ROOT)
PROFILE, PROFILE_CONFIG = execution_profile(CONFIGS)
SEED = int(CONFIGS["training_config"]["seed"])
set_global_seed(SEED)
print({"root": str(ROOT), "profile": PROFILE, "seed": SEED})

{'root': 'D:\\Dropbox\\Work\\Learning\\Python\\toxicity_screening_project', 'profile': 'full', 'seed': 20260723}


In [3]:
from toxicity_screening.utils import atomic_write_json
scope = {
    "title": "Machine Learning-Based Early Toxicity Screening of Small-Molecule Drug Candidates",
    "intended_use": "research-grade early toxicity triage and candidate prioritization",
    "not_intended_use": ["replacement for laboratory testing", "clinical diagnosis", "regulatory decision automation"],
    "input": {"single": "SMILES", "batch": ["molecule_id", "smiles"]},
    "endpoints": list(CONFIGS["endpoints"]["endpoints"]),
    "outputs": ["predicted_class", "raw_probability", "calibrated_probability", "uncertainty", "applicability_domain", "ood_warning", "interpretation", "recommendation"],
    "assumptions": ["small organic molecule parsable by RDKit", "endpoint-specific model bundle exists", "training provenance and splits are available"],
}
atomic_write_json(scope, ROOT / "reports/project_scope.json")
scope

{'title': 'Machine Learning-Based Early Toxicity Screening of Small-Molecule Drug Candidates',
 'intended_use': 'research-grade early toxicity triage and candidate prioritization',
 'not_intended_use': ['replacement for laboratory testing',
  'clinical diagnosis',
  'regulatory decision automation'],
 'input': {'single': 'SMILES', 'batch': ['molecule_id', 'smiles']},
 'endpoints': ['herg_blockade',
  'ames_mutagenicity',
  'SR-p53',
  'SR-ATAD5',
  'SR-ARE',
  'SR-MMP'],
 'outputs': ['predicted_class',
  'raw_probability',
  'calibrated_probability',
  'uncertainty',
  'applicability_domain',
  'ood_warning',
  'interpretation',
  'recommendation'],
 'assumptions': ['small organic molecule parsable by RDKit',
  'endpoint-specific model bundle exists',
  'training provenance and splits are available']}

In [4]:
workflow = """Known molecules and assay labels
→ molecular standardization
→ quality control
→ chemical-space analysis
→ scaffold-aware splitting
→ baseline and advanced modeling
→ endpoint-specific evaluation
→ calibration and uncertainty
→ applicability domain and OOD detection
→ interpretability
→ candidate-screening interface
"""
(ROOT / "reports/workflow.mmd").write_text(workflow, encoding="utf-8")

# Render Matplotlib in the stable base-Anaconda interpreter. This avoids
# loading conflicting OpenMP runtimes inside the toxicity-screening kernel.
import subprocess

plot_python = Path(os.environ.get("TOXICITY_PLOT_PYTHON", r"D:\\Users\\anaconda3\\python.exe"))
if not plot_python.exists():
    raise FileNotFoundError(f"Plotting Python not found: {plot_python}")

plot_worker = r'''
import sys
from pathlib import Path
import matplotlib
matplotlib.use("Agg", force=True)
import matplotlib.pyplot as plt
root = Path(sys.argv[1]).resolve()
workflow = (root / "reports" / "workflow.mmd").read_text(encoding="utf-8")
steps = [line.strip("→ ") for line in workflow.strip().splitlines()]
fig, ax = plt.subplots(figsize=(16, 3.2))
ax.axis("off")
denominator = max(1, len(steps) - 1)
for index, step in enumerate(steps):
    x_position = index / denominator
    ax.text(x_position, 0.55, step.replace(" and ", "\nand ") , ha="center", va="center", fontsize=7, bbox={"boxstyle": "round,pad=0.35", "facecolor": "white", "edgecolor": "black"})
    if index < len(steps) - 1:
        ax.annotate("", xy=((index + 1) / denominator - 0.035, 0.55), xytext=(x_position + 0.035, 0.55), arrowprops={"arrowstyle": "->"})
fig.tight_layout()
output = root / "figures" / "project_workflow.png"
output.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(output, dpi=220, bbox_inches="tight")
plt.close(fig)
print(output)
'''
plot_environment = os.environ.copy()
plot_environment.update({"MPLBACKEND": "Agg", "OMP_NUM_THREADS": "1", "MKL_NUM_THREADS": "1", "OPENBLAS_NUM_THREADS": "1", "NUMEXPR_NUM_THREADS": "1", "VECLIB_MAXIMUM_THREADS": "1", "BLIS_NUM_THREADS": "1"})
completed = subprocess.run([str(plot_python), "-c", plot_worker, str(ROOT)], cwd=str(ROOT), env=plot_environment, capture_output=True, text=True, check=False)
if completed.returncode != 0:
    raise RuntimeError("External workflow plotting failed.\n" f"STDOUT:\n{completed.stdout}\n" f"STDERR:\n{completed.stderr}")
print(completed.stdout.strip())
require_paths([ROOT / "reports/project_scope.json", ROOT / "reports/workflow.mmd", ROOT / "figures/project_workflow.png"])

D:\Dropbox\Work\Learning\Python\toxicity_screening_project\figures\project_workflow.png


### Completion gate
Confirm that the declared artifacts exist before continuing to `01_environment_and_reproducibility.ipynb`.